In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tokenizers import Tokenizer

# 1. Загружаем модель и ее родной токенизатор
model_name = "unsloth/Llama-3.2-1B"
orig_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
orig_tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Загружаем ваш бурятский токенизатор (json)
buryat_json_tokenizer = Tokenizer.from_file("my_buryat_tokenizer.json")

# Извлекаем все текстовые токены из вашего бурятского словаря
buryat_vocab = buryat_json_tokenizer.get_vocab()
# Сортируем по ID, чтобы сохранить внутреннюю логику BPE, и убираем спец-токены
sorted_buryat_tokens = [token for token, _ in sorted(buryat_vocab.items(), key=lambda item: item[1])]
special_tokens_to_skip = ["<s>", "<pad>", "</s>", "<unk>", "<mask>", "Ġ", ""]
new_buryat_words = [t for t in sorted_buryat_tokens if t not in special_tokens_to_skip]

# ================= ИСПРАВЛЕННЫЙ БЛОК (Шаги 3-6) =================

# 3. Добавляем новые токены в оригинальный токенизатор Llama
added_count = orig_tokenizer.add_tokens(new_buryat_words)

# ХИТРОСТЬ: Берём размер старого словаря напрямую ИЗ МАТРИЦЫ ВЕСОВ, а не из токенизатора
old_vocab_size_matrix = orig_model.model.embed_tokens.weight.shape[0] # Строго 128256

# Новый целевой размер — это старая матрица + физически добавленные токены
new_vocab_size = old_vocab_size_matrix + added_count

print(f"Физический размер старой матрицы весов: {old_vocab_size_matrix}")
print(f"Добавлено новых уникальных бурятских токенов: {added_count}")
print(f"Новый физический размер матриц: {new_vocab_size}")

# 4. Расширяем ВХОДНЫЕ эмбеддинги (embed_tokens)
old_embed = orig_model.model.embed_tokens.weight.data
embedding_dim = old_embed.shape[1] # Для Llama 3.2 1B это 2048

new_embed = torch.zeros(new_vocab_size, embedding_dim, dtype=old_embed.dtype, device=old_embed.device)
new_embed[:old_vocab_size_matrix] = old_embed  # Теперь размеры совпадут идеально!

# 5. Расширяем ВЫХОДНЫЕ эмбеддинги (lm_head)
old_lm_head = orig_model.lm_head.weight.data
new_lm_head = torch.zeros(new_vocab_size, embedding_dim, dtype=old_lm_head.dtype, device=old_lm_head.device)
new_lm_head[:old_vocab_size_matrix] = old_lm_head

from tqdm.auto import tqdm

# 6. 🔥 СВЕРХБЫСТРАЯ ИНИЦИАЛИЗАЦИЯ С ШКАЛОЙ ПРОГРЕССА 🔥
print("Инициализация новых бурятских токенов средними весами букв...")

# Заранее вычисляем среднее по всему словарю для ускорения fallback-случаев
global_mean_embed = old_embed.mean(dim=0)
global_mean_lm = old_lm_head.mean(dim=0)

# Оборачиваем цикл в tqdm для наглядного отображения прогресса
for idx, token_str in enumerate(tqdm(new_buryat_words, desc="Обработка токенов")):
    # Физический индекс нового токена в расширенной матрице весов
    matrix_idx = old_vocab_size_matrix + idx

    # Очищаем спецсимволы разметки BPE
    clean_token_str = token_str.replace('Ġ', ' ').strip()

    if clean_token_str:
        # Кодируем старым способом, строго без спецтокенов Llama
        token_ids = orig_tokenizer.encode(clean_token_str, add_special_tokens=False)
        # Оставляем только те ID, которые гарантированно были в оригинальной модели
        token_ids = [tid for tid in token_ids if tid < old_vocab_size_matrix]
    else:
        token_ids = []

    if len(token_ids) > 0:
        # Векторная вставка
        new_embed[matrix_idx] = old_embed[token_ids].mean(dim=0)
        new_lm_head[matrix_idx] = old_lm_head[token_ids].mean(dim=0)
    else:
        # Быстрый fallback без вызова лишних функций
        new_embed[matrix_idx] = global_mean_embed
        new_lm_head[matrix_idx] = global_mean_lm

print("Инициализация успешно завершена!")

# ================================================================


# 7. Перезаписываем матрицы весов в модели
orig_model.model.embed_tokens.weight.data = new_embed
orig_model.lm_head.weight.data = new_lm_head

# 8. Синхронизируем конфиг модели с токенизатором
orig_model.config.vocab_size = new_vocab_size
orig_model.resize_token_embeddings(new_vocab_size) # Встроенный метод Hugging Face для фиксации слоев

# 9. Сохраняем модель И ОБЪЕДИНЕННЫЙ токенизатор вместе
output_dir = "./my_llama_1b_buryat"
orig_model.save_pretrained(output_dir)
orig_tokenizer.save_pretrained(output_dir)

print(f"\nУспешно! Все файлы сохранены в папку '{output_dir}'.")
print("Теперь вы можете использовать эту папку как стартовую точку для Fine-Tuning/Continual Pre-training.")



config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

...

Физический размер старой матрицы весов: 128256
Добавлено новых уникальных бурятских токенов: 15995
Новый физический размер матриц: 144251
Инициализация новых бурятских токенов средними весами букв...


Обработка токенов:   0%|          | 0/15995 [00:00<?, ?it/s]

Инициализация успешно завершена!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Успешно! Все файлы сохранены в папку './my_llama_1b_buryat'.
Теперь вы можете использовать эту папку как стартовую точку для Fine-Tuning/Continual Pre-training.


In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import pandas as pd
df = pd.read_parquet('/content/train-00000-of-00001 (1).parquet')
print(df)

                                                     text
0       Та- мираа барагдажа муудаhан, номхон оолоlion,...
1             Тэ- рээндэ хорхируу үбшэн хүрэhэн бап- гаа.
2       Тэрэ үедэ энэ үбшэнhөо үгытэй грузинска еелени...
3       Сергын хоолой cooiijj  шарха гараад. тэрэ үгэш...
4       Гартаа дүйтэй нэгэ хүгшэн ерэжэ. үбшэн хүбүүнэ...
...                                                   ...
214946  2002 ондо Буряад Уласай Засагай газарай Түрүүл...
214947  2004 онһоо 2017 он болотор Буряад Уласай Засаг...
214948  Уданшьегүй Арадай Хуралай бүридхэлдэ орожо, та...
214949  Мүн баһа Иннокентий Матвеевич Бүгэдэ Буряадай ...
214950  Автор: Сэнгэ РИНЧИНОВ буряадшалаа Фото: Буряад...

[214951 rows x 1 columns]


In [ ]:
from datasets import Dataset

# 4. Подготовка датасета напрямую ИЗ ВАШЕГО PANDAS DATAFRAME
print("Конвертация датасета из Pandas DataFrame в формат Hugging Face...")

# Накануне убедимся, что переменная вашего датафрейма называется df.
# Если у вас другое имя переменной (например, my_df), замените df на него ниже.
# Фильтруем пустые строки и приводим всё к текстовому формату
df_clean = df.dropna(subset=['text'])
df_clean['text'] = df_clean['text'].astype(str)
# Удаляем полные дубликаты текстов, которые могут вызывать коллапс модели


# Переводим напрямую из памяти
dataset = Dataset.from_pandas(df_clean)

print(f"Успешно загружено {len(dataset)} строк для обучения!")


Конвертация датасета из Pandas DataFrame в формат Hugging Face...
Успешно загружено 214951 строк для обучения!


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

# 1. Пути к данным и сохраненной ранее модели
model_path = "./my_llama_1b_buryat"  # Папка из предыдущего шага, где лежат веса и токенизатор
corpus_file = "/content/train-00000-of-00001 (1).parquet"           # Ваш текстовый файл только на бурятском языке
output_dir = "./cpt_llama_buryat_result"


import codecs

source_file = "/content/train-00000-of-00001 (1).parquet"
temp_file = "corpus_fixed_utf8.txt"

# Поочередно проверяем самые вероятные кодировки для такой ошибки
possible_encodings = ['utf-16', 'utf-16-le', 'utf-16-be', 'utf-8-sig']

success = False
for enc in possible_encodings:
    try:
        print(f"Пробуем прочитать файл в кодировке: {enc}...")
        with open(source_file, 'r', encoding=enc) as f:
            content = f.read()

        # Если прочиталось без ошибок — сразу сохраняем в правильный UTF-8
        with open(temp_file, 'w', encoding='utf-8') as f_out:
            f_out.write(content)

        success = True
        print(f"🎉 Успех! Файл успешно расшифрован с помощью {enc} и перекодирован в UTF-8.")
        break
    except UnicodeDecodeError:
        print(f"❌ Кодировка {enc} не подошла, ищем дальше...")

if success:
    # Заменяем старый плохой файл новым хорошим
    import os
    os.remove(source_file)
    os.rename(temp_file, source_file)
    print("Файл 'corpus.txt' обновлен и готов к обучению!")
else:
    print("⚠️ Не удалось автоматически определить кодировку. Проверьте, не поврежден ли файл физически.")

# 2. Загрузка расширенного токенизатора и модели
print("Загрузка модели и токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
# Устанавливаем pad_token, если он не задан (для корректного батчинга)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,       # Обучаем в полуточности fp16 для экономии VRAM
    device_map="auto"                # Автоматически задействует GPU
)
# Фиксируем масштаб градиентов для слоя эмбеддингов
model.get_input_embeddings().weight.data.mul_(0.1)

# 3. 🧠 ХИТРАЯ ЗАМОРОЗКА СЛОЕВ (Фильтруем градиенты) 🧠
print("Применяем частичную заморозку весов...")
model.config.use_cache = False  # Отключаем кэш для корректного расчета градиентов при обучении

# Находим индекс последнего слоя модели (для Llama 3.2 1B их обычно 16, индексы 0-15)
num_layers = len(model.model.layers)
trainable_layer_indices = list(range(num_layers - 4, num_layers)) # Последние 4 слоя
print(f"Всего слоев в модели: {num_layers}. Будут обучаться слои: {trainable_layer_indices}")

# Сначала замораживаем абсолютно все параметры
for param in model.parameters():
    param.requires_grad = False

# Размораживаем только то, что указано в ТЗ:
# 1. Входные эмбеддинги
model.model.embed_tokens.weight.requires_grad = True

# 2. Выходная языковая голова lm_head
model.lm_head.weight.requires_grad = True

# 3. Последние 4 слоя блоков Attention
for idx in trainable_layer_indices:
    for param in model.model.layers[idx].parameters():
        param.requires_grad = True

# Проверяем, сколько параметров осталось активными для обучения
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Обучаемые параметры: {trainable_params:,} из {all_params:,} ({100 * trainable_params / all_params:.2f}%)")

# 4. Подготовка датасета (Causal LM формат для сырого текста)
print("Загрузка и подготовка текстового корпуса...")
# Загружаем простой текстовый файл как датасет Hugging Face
# Заменяем старую строку загрузки на эту (с явным указанием кодировки):
# Возвращаем стандартную очищенную строку загрузки датасета:
#dataset = load_dataset("text", data_files={"train": corpus_file})
dataset = dataset




training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # --- СНИЖАЕМ СКОРОСТЬ ДЛЯ ЗАЩИТЫ ОТ КОЛЛАПСА МОДЕЛИ ---
    learning_rate=4e-5,                  # Снижено с 2e-4 для плавного изучения эмбеддингов
    num_train_epochs=1,                  # СТРОГО 1 эпоха! Больше для CPT делать вредно
    # ------------------------------------------------------

    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,                   # Увеличиваем разогрев до 6% для плавного старта
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,

    # --- КРИТИЧЕСКИ ВАЖНЫЙ ПАРАМЕТР КЛИППИНГА ГРАДИЕНТОВ ---
    max_grad_norm=1.0,                   # Срезает аномальные пики градиентов, спасая от 0.00
    # -------------------------------------------------------

    report_to="none"
)


# Хак для стабильности градиентов размороженных эмбеддингов
model.enable_input_require_grads()

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,               # Ваш Dataset из Pandas
    args=training_args                    # Склеивает строки, сокращая общее число шагов в разы
)

# 6. Запуск SFTTrainer в режиме Continued Pretraining
# Использование SFTTrainer удобно тем, что он автоматически упаковывает



# 7. Старт обучения
print("Старт дообучения Continued Pretraining...")
trainer.train()

# 8. Финальное сохранение полностью готовой бурятской модели
print("Обучение завершено. Сохранение финальных весов...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Модель успешно сохранена в '{output_dir}'! Теперь она понимает бурятский язык.")


Пробуем прочитать файл в кодировке: utf-16...
❌ Кодировка utf-16 не подошла, ищем дальше...
Пробуем прочитать файл в кодировке: utf-16-le...
❌ Кодировка utf-16-le не подошла, ищем дальше...
Пробуем прочитать файл в кодировке: utf-16-be...
❌ Кодировка utf-16-be не подошла, ищем дальше...
Пробуем прочитать файл в кодировке: utf-8-sig...
❌ Кодировка utf-8-sig не подошла, ищем дальше...
⚠️ Не удалось автоматически определить кодировку. Проверьте, не поврежден ли файл физически.
Загрузка модели и токенизатора...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Применяем частичную заморозку весов...
Всего слоев в модели: 16. Будут обучаться слои: [12, 13, 14, 15]
Обучаемые параметры: 538,712,064 из 1,268,572,160 (42.47%)
Загрузка и подготовка текстового корпуса...


...

Tokenizing train dataset:   0%|          | 0/214951 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/214951 [00:00<?, ? examples/s]

Старт дообучения Continued Pretraining...


Step,Training Loss
10,11.353967
20,11.100780
30,10.752767
40,11.025595
50,13.058186
60,14.928256
70,24.242104
80,23.987148
90,25.491364
100,24.613289


KeyboardInterrupt: 

In [ ]:
print("Обучение завершено. Сохранение финальных весов...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Модель успешно сохранена в '{output_dir}'! Теперь она понимает бурятский язык.")

Обучение завершено. Сохранение финальных весов...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель успешно сохранена в './cpt_llama_buryat_result'! Теперь она понимает бурятский язык.


In [ ]:
import pandas as pd
df = pd.read_parquet('/content/train-00000-of-00001 (1).parquet')
print(df)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

import os

# Заменяем относительный путь "./cpt_base_buryat_model" на абсолютный путь в Colab
model_path = os.path.abspath("/content/cpt_base_buryat_model")
output_dir = os.path.abspath("/content/buryat_llama_instruction_final")

print(f"Используем абсолютный локальный путь к модели: {model_path}")


# 2. Подготовка и разметка диалогов (Формат Llama 3/3.2 Chat)
def format_prompts(batch):
    formatted_texts = []
    for instruct, response in zip(batch['ru'], batch['bua']):
        # Применяем официальный шаблон разметки диалогов Llama 3
        text = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruct}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Форматирование инструктивного датасета...")
# Конвертируем ваш Pandas DataFrame (df_instruct) в формат Hugging Face
# Очищаем от пустых строк, если они есть
df_instruct = pd.read_parquet('/content/train-00000-of-00001-35919150bc94e0c2 (1).parquet')
df_instruct_clean = df_instruct.dropna(subset=['ru', 'bua']).astype(str)
instruct_dataset = Dataset.from_pandas(df_instruct_clean)
instruct_dataset = instruct_dataset.map(format_prompts, batched=True)
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Указываем точный путь к папке, где лежит config.json
model_path = "/content/cpt_base_buryat_model"

print("Загрузка локального токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True  # !!! СТРОГО ИЩЕМ ТОЛЬКО НА ЛОКАЛЬНОМ ДИСКЕ !!!
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Конфигурация QLoRA остаётся прежней...
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Загрузка локальной квантованной модели...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True  # !!! СТРОГО ИЩЕМ ТОЛЬКО НА ЛОКАЛЬНОМ ДИСКЕ !!!
)


# Подготавливаем архитектуру квантованной модели к обучению градиентов
model = prepare_model_for_kbit_training(model)

# 5. Настройка LoRA-параметров по вашему ТЗ
print("Конфигурация LoRA адаптеров...")
peft_config = LoraConfig(
    r=64,                                 # СТРОГО ПО ТЗ (Rank = 64)
    lora_alpha=128,                       # Стандартное масштабирование (обычно r * 2)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # СТРОГО ПО ТЗ
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Оборачиваем базовую модель в LoRA-матрицы
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()       # Выведет, что обучается всего ~2-3% параметров

# 6. Конфигурация параметров обучения (SFT Arguments)
# На этапе QLoRA память расходуется очень экономно, можно увеличить физический батч
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,        # Можно ставить 2 или 4, так как QLoRA не ест память
    gradient_accumulation_steps=4,        # Итоговый стабильный батч равен 2 * 4 = 8
    learning_rate=2e-4,                   # Стандартный LR для этапа инструкций
    num_train_epochs=3,                   # Инструкции обычно учат от 2 до 5 эпох
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",                # Сохраняем модель в конце каждой эпохи
    save_total_limit=1,                   # Храним только последний лучший чекпоинт
    fp16=True,                            # Включаем fp16 (в QLoRA с LoRA это абсолютно стабильно)
    optim="paged_adamw_32bit",            # Специальный оптимизатор для QLoRA (защищает от OOM)
    gradient_checkpointing=True,
    report_to="none"
)

# 7. Запуск SFTTrainer
# Для инструкций параметр packing ставим в False, чтобы модель четко видела границы диалогов
trainer = SFTTrainer(
    model=model,
    train_dataset=instruct_dataset,
    dataset_text_field="text",
    max_seq_length=1024,                  # 1024 токенов обычно с запасом хватает для пар вопрос-ответ
    tokenizer=tokenizer,
    args=training_args,
    packing=False                         # СТРОГО False для инструктивного обучения
)

print("Старт инструктивного дообучения QLoRA...")
trainer.train()

# 8. Сохранение финальных LoRA-адаптеров
print("Обучение завершено! Сохранение адаптеров...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Финал! Инструктивные веса сохранены в '{output_dir}'.")


Используем абсолютный локальный путь к модели: /content/cpt_base_buryat_model
Форматирование инструктивного датасета...


Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Загрузка локального токенизатора...


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/cpt_base_buryat_model'. Use `repo_type` argument if needed.

In [ ]:
import os

paths_to_check = [
    "./cpt_base_buryat_model",
    "/content/cpt_base_buryat_model",
    "./cpt_llama_buryat_result",
    "/content/cpt_llama_buryat_result",
]

found = False
for p in paths_to_check:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p):
        print(f"✅ Папка НАЙДЕНА: {abs_p}")
        print(f"Содержимое папки: {os.listdir(abs_p)}")
        found = True
        break

if not found:
    print("❌ Ни одна из папок не найдена на диске!")
    print("Текущие папки в директории /content/:", os.listdir("/content/"))


✅ Папка НАЙДЕНА: /content/cpt_llama_buryat_result
Содержимое папки: ['generation_config.json', 'config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
k = pd.read_parquet('/content/train-00000-of-00001-35919150bc94e0c2 (1).parquet')
print(k)

                                                     ru  \
0                                        ПОСТАНОВЛЕНИЕ.   
1                                          г. Улан-Удэ.   
2     О внесении изменений в постановление Правитель...   
3     В целях приведения правового акта Правительств...   
4     Внести следующие изменения в постановление Пра...   
...                                                 ...   
1327  Внести следующие изменения в Перечень мероприя...   
1328  Строки 1.102 и 1.103 изложить в следующей реда...   
1329  Дополнить строками 1.106 - 1.115 следующего со...   
1330  Настоящее постановление вступает в силу со дня...   
1331  Проект представлен Министерством здравоохранен...   

                                                    bua  
0                                              ТОГТООЛ.  
1                                       Улаан-Үдэ хото.  
2     «Хэлсээнэй сэн хубилгаха (дээшэлүүлхэ) арга бо...  
3     Буряад Уласай Засагай газарай хуулита шиидхэбэ...  
4

In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

# 1. 🎯 УКАЗЫВАЕМ ТОЧНЫЙ НАЙДЕННЫЙ ПУТЬ К МОДЕЛИ
model_path = "/content/cpt_llama_buryat_result"  # Путь, где лежат ваши файлы CPT
output_dir = "/content/buryat_llama_instruction_final"

# 2. Подготовка и разметка диалогов (Формат Llama 3/3.2 Chat)
def format_prompts(batch):
    formatted_texts = []
    for instruct, response in zip(batch['ru'], batch['bua']):
        # Применяем официальный шаблон разметки диалогов Llama 3
        text = (
            f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruct}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

print("Форматирование инструктивного датасета...")
# Конвертируем ваш Pandas DataFrame (df_instruct) в формат Hugging Face
df_instruct = pd.read_parquet('/content/train-00000-of-00001-35919150bc94e0c2 (1).parquet')
df_instruct_clean = df_instruct.dropna(subset=['ru', 'bua']).astype(str)
instruct_dataset = Dataset.from_pandas(df_instruct_clean)
instruct_dataset = instruct_dataset.map(format_prompts, batched=True)

# 3. Настройка 4-битного квантования (QLoRA)
# 3. Настройка 4-битного квантования (Обновлено для стабильности на GPU T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32, # ИСПРАВЛЕНО: float32 вместо float16
    bnb_4bit_use_double_quant=True,
)


# 4. Загрузка токенизатора и квантованной базовой модели СТРОГО ЛОКАЛЬНО
print("Загрузка локального токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True                # Запрещаем искать в интернете
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Загрузка локальной квантованной модели...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",                    # Автоматически сажаем на GPU
    local_files_only=True                # Запрещаем искать в интернете
)

# Подготавливаем архитектуру квантованной модели к обучению градиентов
model = prepare_model_for_kbit_training(model)

# 5. Настройка LoRA-параметров по вашему ТЗ
print("Конфигурация LoRA адаптеров...")
peft_config = LoraConfig(
    r=64,                                 # СТРОГО ПО ТЗ (Rank = 64)
    lora_alpha=128,                       # Стандартное масштабирование (r * 2)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # СТРОГО ПО ТЗ
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Оборачиваем базовую модель в LoRA-матрицы
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()       # Выведет процент обучаемых весов

# 6. Конфигурация параметров обучения (SFT Arguments)
# 6. Конфигурация параметров обучения (SFT Arguments)
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,

    # --- ИСПРАВЛЕНО ДЛЯ ИСКЛЮЧЕНИЯ ОШИБКИ BFLOAT16 ---
    fp16=False,                           # СТРОГО False! Отключаем падающий GradScaler
    bf16=False,                           # СТРОГО False для видеокарт поколения T4
    # -------------------------------------------------

    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    report_to="none"
)


# 7. Запуск SFTTrainer
# Железобетонный вариант (включить packing=True):
trainer = SFTTrainer(
    model=model,
    train_dataset=instruct_dataset,

    args=training_args
)


print("Старт инструктивного дообучения QLoRA...")
trainer.train()

# 8. Сохранение финальных LoRA-адаптеров
print("Обучение завершено! Сохранение адаптеров...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Финал! Инструктивные LoRA-веса сохранены в '{output_dir}'.")


Форматирование инструктивного датасета...


Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Загрузка локального токенизатора...
Загрузка локальной квантованной модели...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

Tokenizing train dataset:   0%|          | 0/1332 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1332 [00:00<?, ? examples/s]

Старт инструктивного дообучения QLoRA...


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


KeyboardInterrupt: 

In [ ]:
# 8. Сохранение финальных LoRA-адаптеров
print("Обучение завершено! Сохранение адаптеров...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Финал! Инструктивные LoRA-веса сохранены в '{output_dir}'.")

Обучение завершено! Сохранение адаптеров...
Финал! Инструктивные LoRA-веса сохранены в '/content/buryat_llama_instruction_final'.


In [ ]:
import torch
from transformers import pipeline

# 1. Если вы тестируете модель СРАЗУ ПОСЛЕ ОБУЧЕНИЯ (модель и токенизатор в памяти):
# Мы переводим модель в режим оценки (evaluation mode)
model.eval()

# 2. Создаем пайплайн генерации текста
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

# 3. Пишем запрос (Prompt) на бурятском языке.
# Используем тот же шаблон разметки Llama 3 Chat, на котором обучали модель на Шаге 5
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты." # Запрос: "Назовите столицу Республики Бурятия."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

# 4. Запускаем генерацию
print("Модель думает над ответом...")
outputs = generator(
    prompt,
    max_new_tokens=100,        # Максимальная длина ответа (в токенах)
    do_sample=True,            # Включаем случайный выбор слов для «живой» речи
    temperature=0.6,           # Чем ниже (например, 0.3), тем точнее ответ. Чем выше (0.8), тем креативнее.
    top_p=0.9,                 # Ограничение выборки слов по вероятности
    eos_token_id=tokenizer.eos_token_id
)

# 5. Выводим чистый результат
generated_text = outputs[0]['generated_text']
print("\n--- ОТВЕТ МОДЕЛИ ---")
print(generated_text)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'do_sample', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Модель думает над ответом...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 1. Точные пути к вашим папкам
base_cpt_path = "/content/cpt_llama_buryat_result"         # Папка после CPT (Шаг 4)
lora_weights_path = "/content/buryat_llama_instruction_final" # Папка после LoRA (Шаг 5)

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path, local_files_only=True)

# 2. Конфигурация квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

print("2. Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_cpt_path,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True
)

# 🔥 ЖЕСТКИЙ ФИКС ДЛЯ ЗАЩИТЫ ОТ CUDA error:
# Принудительно расширяем размер эмбеддингов базовой модели до размера токенизатора (143 823)
# Это создаст недостающие строки в матрице весов и предотвратит вылет ассерта
print(f"Фикс размера словаря: {base_model.config.vocab_size} -> {len(tokenizer)}")
base_model.resize_token_embeddings(len(tokenizer))

print("3. Накатывание LoRA-адаптеров инструкций...")
# Теперь безопасно загружаем адаптеры
model = PeftModel.from_pretrained(base_model, lora_weights_path)
model.eval() # Переводим в режим инференса

# 4. Создаем пайплайн генерации текста
from transformers import pipeline
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

# Шаблон разметки Llama 3 Chat
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

print("\nМодель генерирует ответ...")
outputs = generator(
    prompt,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.4,           # Немного снизим температуру для более точного ответа
    top_p=0.9,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

print("\n--- ОТВЕТ МОДЕЛИ ---")
print(outputs[0]['generated_text'])


1. Загрузка расширенного токенизатора...
2. Загрузка базовой модели...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] LlamaForCausalLM LOAD REPORT from: /content/cpt_llama_buryat_result
Key                                           | Status     |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

RuntimeError: We encountered some issues during automatic conversion of the weights. For details look at the `CONVERSION` entries of the above report!

In [ ]:
!pip install -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from transformers import pipeline

# 1. Берем пути
# Используем папку из Шага 5 для токенизатора и LoRA
lora_weights_path = "/content/buryat_llama_instruction_final"
# Если папка CPT повреждена, мы можем временно указать путь к исходной расширенной модели из Шага 3,
# чтобы проверить, как работают LoRA-инструкции
base_model_path = "/content/my_llama_1b_buryat"

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path, local_files_only=True)

print("2. Загрузка базовой модели в безопасном режиме FP16 (Без квантования)...")
# Мы отключаем BitsAndBytesConfig, чтобы битые или несовместимые слои не вешали CUDA
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16, # Чистый полуточный режим
    device_map="auto",
    local_files_only=True
)

# Принудительно выставляем правильный размер матриц
base_model.resize_token_embeddings(len(tokenizer))

print("3. Накатывание LoRA-адаптеров инструкций...")
model = PeftModel.from_pretrained(base_model, lora_weights_path)
model.eval()

# 4. Создаем пайплайн генерации текста
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

# Шаблон разметки Llama 3 Chat
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

print("\nМодель генерирует ответ...")
with torch.no_grad(): # Отключаем расчет градиентов для стабильности инференса
    outputs = generator(
        prompt,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- ОТВЕТ МОДЕЛИ ---")
print(outputs['generated_text'])


1. Загрузка расширенного токенизатора...
2. Загрузка базовой модели в безопасном режиме FP16 (Без квантования)...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

3. Накатывание LoRA-адаптеров инструкций...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

# 1. Задаем пути к исходной чистой модели и адаптерам
base_model_path = "/content/my_llama_1b_buryat"        # Гарантированно целая база из Шага 3
lora_weights_path = "/content/buryat_llama_instruction_final" # Ваши LoRA-адаптеры из Шага 5

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path, local_files_only=True)

print("2. Загрузка базовой модели строго на CPU...")
# Загружаем модель в стандартной точности float32 на CPU для абсолютной стабильности
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float32,     # float32 исключает любые проблемы с типами данных
    device_map={"": "cpu"},        # Явно принуждаем модель оставаться в оперативной памяти (RAM)
    local_files_only=True
)

# Выравниваем размер эмбеддингов до применения LoRA
base_model.resize_token_embeddings(len(tokenizer))

print("3. Накатывание LoRA-адаптеров (на CPU)...")
# Подгружаем сохраненные LoRA-инструкции
model = PeftModel.from_pretrained(base_model, lora_weights_path, device_map={"": "cpu"})
model.eval()

print("4. Создание пайплайна генерации текста...")
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1                      # Флаг -1 принудительно заставляет pipeline работать на CPU
)

# Официальный шаблон разметки диалогов Llama 3 Chat
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты." # Ваш запрос к модели
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

print("\nМодель генерирует ответ (это может занять до 15 секунд на CPU)...")
with torch.no_grad():
    outputs = generator(
        prompt,
        max_new_tokens=40,         # Длина ответа
        do_sample=True,
        temperature=0.3,           # Низкая температура для минимизации галлюцинаций
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- ОТВЕТ МОДЕЛИ ---")
print(outputs['generated_text'])


1. Загрузка расширенного токенизатора...
2. Загрузка базовой модели строго на CPU...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

3. Накатывание LoRA-адаптеров (на CPU)...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Указываем пути к вашим локальным файлам в Colab
lora_weights_path = "/content/buryat_llama_instruction_final" # Папка из Шага 5

# 2. Загружаем токенизатор и конфигурацию LoRA из локальной папки
print("Загрузка локальных файлов перед отправкой...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path)
model = AutoModelForCausalLM.from_pretrained(lora_weights_path)

# 3. Придумываем название репозитория на Hugging Face
# Формат: "ваш_никнейм/название_модели"
hf_repo_id = "poco28/llama-3.2-1b-buryat-lora"

print(f"Отправка файлов в репозиторий {hf_repo_id}...")

# 4. Публикуем токенизатор (зальет tokenizer.json с бурятскими буквами)
tokenizer.push_to_hub(hf_repo_id, private=True) # private=True сделает модель видимой только вам

# 5. Публикуем сами LoRA веса
model.push_to_hub(hf_repo_id, private=True)

print(f"🎉 Успешно! Модель доступна по ссылке: https://huggingface.com{hf_repo_id}")


Загрузка локальных файлов перед отправкой...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/128 [00:00<?, ?it/s]

Отправка файлов в репозиторий poco28/llama-3.2-1b-buryat-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 27.3MB / 27.3MB            

🎉 Успешно! Модель доступна по ссылке: https://huggingface.copoco28/llama-3.2-1b-buryat-lora


In [ ]:
import torch
import os

# 1. Жестко запрещаем PyTorch и Safetensors вообще видеть видеокарту на системном уровне
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Папки из прошлых шагов
base_model_path = "/content/my_llama_1b_buryat"        # База из Шага 3
lora_weights_path = "/content/buryat_llama_instruction_final" # Адаптеры из Шага 5

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path)

print("2. Загрузка базовой модели на CPU (в float32 для стабильности)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

# Выравниваем размер матриц строго до сборки
base_model.resize_token_embeddings(len(tokenizer))

print("3. Сборка LoRA-адаптеров с базой (строго в RAM)...")
# Подгружаем веса адаптеров
model = PeftModel.from_pretrained(base_model, lora_weights_path)
model.eval()

print("4. Перевод модели в монолитный режим генерации...")
# Объединяем слои на CPU, чтобы убрать лишние прослойки вызовов peft
model = model.merge_and_unload()

# 5. Ручной запуск генерации без использования тяжелых пайплайнов
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

inputs = tokenizer(prompt, return_tensors="pt")

print("\nМодель генерирует текст на CPU в защищенном режиме...")

# Извлекаем attention_mask, который обязателен для корректной работы Llama 3
attention_mask = inputs.get("attention_mask", None)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=attention_mask,
        max_new_tokens=40,

        # 🔥 ОТКЛЮЧАЕМ СЭМПЛИРОВАНИЕ ДЛЯ ИСКЛЮЧЕНИЯ NAN/INF ОШИБКИ 🔥
        do_sample=False,             # Жадный поиск (Greedy Search) вместо multinomial

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Декодируем результат
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- ОТВЕТ МОДЕЛИ ---")
print(generated_text)


1. Загрузка расширенного токенизатора...
2. Загрузка базовой модели на CPU (в float32 для стабильности)...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Возвращаем пути к вашим папкам
base_model_path = "/content/my_llama_1b_buryat"        # База из Шага 3
lora_weights_path = "/content/buryat_llama_instruction_final" # Адаптеры из Шага 5

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path)

print("2. Загрузка базовой модели на GPU в режиме FP16 (Экономим RAM)...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,     # float16 весит всего 2.5 ГБ, RAM не переполнится
    device_map="auto"              # Сажаем модель строго в видеопамять GPU
)

# Выравниваем размер матриц строго под токенизатор
model.resize_token_embeddings(len(tokenizer))

print("3. Накатывание LoRA-адаптеров в видеопамять...")
# Подгружаем веса адаптеров поверх базы без слияния слоев (merge_and_unload убран)
model = PeftModel.from_pretrained(model, lora_weights_path)
model.eval()

# 4. Подготовка текста
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "кто ты"
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

# Переносим входные токены на GPU к модели
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\nМодель генерирует текст на GPU...")
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask", None),
        max_new_tokens=40,

        # Защита от NaN ошибок: жесткий выбор лучшего токена без multinomial
        do_sample=False,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Декодируем результат
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- ОТВЕТ МОДЕЛИ ---")
print(generated_text)


1. Загрузка расширенного токенизатора...
2. Загрузка базовой модели на GPU в режиме FP16 (Экономим RAM)...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

3. Накатывание LoRA-адаптеров в видеопамять...


[transformers] Both `max_new_tokens` (=40) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Модель генерирует текст на GPU...

--- ОТВЕТ МОДЕЛИ ---
<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>

кто ты<|eot_id|><|start_header_id|>assistant<|end_header_id|>

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [ ]:
print("\nМодель генерирует текст в режиме подавления повторов...")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask", None),
        max_new_tokens=50,

        # 🔥 СНИМАЕМ ЗАКЛИНИВАНИЕ ВЕСОВ 🔥
        do_sample=True,              # Возвращаем живой выбор слов
        temperature=0.4,             # Невысокая креативность для точности
        top_p=0.85,                  # Берем только топ вероятных слов

        # 🎯 ШТРАФ ЗА ПОВТОРЫ (Если модель напишет один "!", второй раз ей будет запрещено это делать)
        repetition_penalty=1.3,      # Значения от 1.2 до 1.5 жестко ломают циклы повторений
        no_repeat_ngram_size=3,      # Запрещает повторять одинаковые цепочки из 3 токенов

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Декодируем и выводим
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- ОБНОВЛЕННЫЙ ОТВЕТ МОДЕЛИ ---")
print(generated_text)


[transformers] Both `max_new_tokens` (=50) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Модель генерирует текст в режиме подавления повторов...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. МЕНЯЕМ БАЗУ НА ГАРАНТИРОВАННО ЧИСТУЮ ИЗ ШАГА 3
base_clean_path = "/content/my_llama_1b_buryat"          # Исходная расширенная модель БЕЗ битых CPT весов
lora_weights_path = "/content/buryat_llama_instruction_final" # Ваши сохраненные LoRA-адаптеры

print("1. Загрузка расширенного токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(lora_weights_path)

print("2. Загрузка ЧИСТОЙ базовой модели на GPU в режиме FP16...")
# Загружаем исходную гибридную модель, в которой нет NaN и бесконечностей
model = AutoModelForCausalLM.from_pretrained(
    base_clean_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Выравниваем размер матриц строго под токенизатор (143 823)
model.resize_token_embeddings(len(tokenizer))

print("3. Накатывание LoRA-адаптеров инструкций...")
# Применяем ваши бурятские LoRA-веса к чистой базе
model = PeftModel.from_pretrained(model, lora_weights_path)
model.eval()

# 4. Подготовка текста запроса
prompt = (
    "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
    "Буряад Уласай нийслэл хото нэрлэгты." # "Назовите столицу Республики Бурятия."
    "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("\nМодель генерирует ответ...")
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask", None),
        max_new_tokens=50,

        # Строго выключаем сэмплирование для защиты от CUDA assert
        do_sample=False,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Декодируем и смотрим результат
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- ОТВЕТ МОДЕЛИ ---")
print(generated_text)


1. Загрузка расширенного токенизатора...
2. Загрузка ЧИСТОЙ базовой модели на GPU в режиме FP16...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

3. Накатывание LoRA-адаптеров инструкций...


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch
import os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Точные пути
model_path = "/content/my_llama_1b_buryat"  # Работаем от надежной исходной базы
output_dir = "/content/buryat_llama_instruction_FIXED"

# Форматирование датасета (Шаблон Llama 3)
def format_prompts(batch):
    formatted_texts = []
    for instruct, response in zip(batch['ru'], batch['bua']):
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruct}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"
        formatted_texts.append(text)
    return {"text": formatted_texts}
df_instruct = pd.read_parquet('/content/train-00000-of-00001-35919150bc94e0c2 (1).parquet')
df_instruct_clean = df_instruct.dropna(subset=['ru', 'bua']).astype(str)
instruct_dataset = Dataset.from_pandas(df_instruct_clean).map(format_prompts, batched=True)

# 4-битное квантование в float32 (Для стабильности на GPU T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path, quantization_config=bnb_config, device_map="auto", local_files_only=True
)
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA по ТЗ (Rank 64)
peft_config = LoraConfig(
    r=64, lora_alpha=128,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Ультра-безопасные аргументы обучения
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,                   # Чуть снизим LR для идеальной точности
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="no",
    fp16=False, bf16=False,               # Отключаем fp16 градиенты, пишем в float32
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    max_grad_norm=0.5,                    # Жесткий предохранитель от взрыва весов
    report_to="none"
)

# 🔥 ГЛАВНЫЙ СЕКРЕТ: Упаковываем через SFTConfig
# packing=True принудительно уберет пустые маски ответов и уничтожит нули в лоссе!
sft_config = SFTConfig(                 # 512 токенов защитит от OOM
    packing=True,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model, train_dataset=instruct_dataset, args=training_args
)

print("Старт чистого инструктивного дообучения QLoRA...")
trainer.train()

# Сохраняем новые, рабочие адаптеры
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Готово! Новые адаптеры сохранены в паку buryat_llama_instruction_FIXED")


Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/1332 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1332 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1332 [00:00<?, ? examples/s]

Старт чистого инструктивного дообучения QLoRA...


Step,Training Loss
5,8.028548
